Problem Statement (Canonical Form)

You are given a list of currency exchange rates like:

USD → EUR = 0.9
EUR → GBP = 0.8
GBP → USD = 1.5

Question:
Is there a sequence of trades that starts with some currency and ends with more money than you started with?

👉 i.e., can you make profit without risk? (arbitrage opportunity)

In [ ]:
# Solution
# Try to check if there is a negative cycle between any set of ccy pairs
# If it has negative wt cycle it means the restantant wt after each cycle is moving towards negative infinity
# hence there is an arbitrage opportunity

# Here the problem is not only to find if there is ccy arbitrage but we need to check for each ccy as the starting node
# to find if arbitrage opportunity is present from this ccy

# Steps
# Create ccy set to get unique ccy pair
# map ccy to ids 
# create edges with weights as negative log of rate
# Run bellman ford algo over it to find negative weighted cycle


import math
def currency_arbitrage(rates):
    ccys = set()

    for u, v, _ in rates:
        ccys.add(u)
        ccys.add(v)
    
    n = len(ccys)
    ccys_to_ids = {ccy: id for id, ccy in enumerate(ccys)}

    edges = []
    for u, v, rate in rates:
        u_id = ccys_to_ids[u]
        v_id = ccys_to_ids[v]
        # why log, because shortest distance allows summation, 
        # for profit r1 * r2 * r3 > 1 we need to convert it to log[r1] + log[r2] + log[r3] > log[0]
        # why negative log? 
        # Because shortest distance is for global minima but here we want maxima hence flipping the sign 
        log_rate = -math.log(rate) 
        edges.append((u_id, v_id, log_rate))
    
    distance = [float('inf')]*n
    distance[0] = 0

    # why n-1 for loop
    # because in bellman we try to relax all edges from each node to every other node
    # since there are n node so we loop over n-1 edges max
    for _ in range(n-1):
        for u, v, wt in edges:
            # why distance[u] + wt < distance[v]
            # This is not only for bellman but any shortest distance algo like dijkstra's
            # here we are checking if there is a cheaper way to reach v from u compared to existing cost
            # eg - A -> B 100, B -> C 200, A -> C 500
            # Traversal from A to C
            # AB 100, AC 500, A->B->C 300 hence we update the cost of A->C to 300
            if distance[u] + wt < distance[v]:
                distance[v] = distance[u] + wt
    
    # why loop again
    # this tells if there is a negative cycle or not
    for u, v, wt in edges:
            if distance[u] + wt < distance[v]:
                return True
    
    return False

currency_arbitrage([['USD', 'EUR', 0.9], ['EUR', 'GBP', 0.8], ['GBP', 'USD', 1.2]])